In [ ]:
'''
notebook for creating salinity normalized model variables
'''

In [ ]:
import warnings
warnings.filterwarnings('ignore') # don't output warnings
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from pathlib import Path
import glob
import pandas as pd 

In [ ]:
data_dir = '/space/hall7/sitestore/eccc/crd/cccma/users/rpg002/data/'
runs = ['assimilation', 'historical']
models = ['CanESM5','CanESM5-CanOE']
variables_to_normalize = ['talk', 'dissic']

In [ ]:
for run in runs:
    data_sal = xr.open_mfdataset(data_dir + f'so/{run}/CanESM5/*.nc', combine = 'nested', concat_dim = 'time')['so']

    for model in models:

        for var in variables_to_normalize:
            if len(glob.glob(data_dir + f'n{var}/{run}/{model}/*.nc')) == 0:
                print( run , model)

                if all([run == 'assimilation', model == 'CanESM5-CanOE']):
                    model = 'CanESM5-CanOE_1'
                
                data_var = xr.open_mfdataset(data_dir + f'{var}/{run}/{model}/*.nc', combine = 'nested', concat_dim = 'time')[var]
                if all([run == 'assimilation', model == 'CanESM5-CanOE_1']):
                    data_var['ensembles'] = np.array([ f'r{i}i1p2f1' for i in data_var['ensembles'].values])

                data_nvar = data_var * (35/data_sal)

                y0 = data_nvar.time.min().values.item().year
                y1 = data_nvar.time.max().values.item().year
                data_dir_out = data_dir + f'n{var}/{run}/{model}'
                Path(data_dir_out).mkdata_dir(parents=True, exist_ok=True)
                display(data_nvar.to_dataset(name = var).sortby('ensembles').transpose('ensembles','time',...,'lat','lon'))
                data_nvar.to_dataset(name = var).sortby('ensembles').transpose('ensembles','time',...,'lat','lon').to_netcdf(f'{data_dir_out}/n{var}_Omon_ensmebles_{y0}01_{y1}12_1x1_LE.nc' )